# Exercise 2 — Clean the keys

**Learner exercise** · [All exercises](../index.html) · [Setup](../README.md)

## What you’ll learn

- Compose built-in Spark functions to normalise product keys.
- Reuse a Column expression in DataFrame transformations while keeping the raw inputs available.

**Core: about 5 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 5 extra minutes; choose it here if the topic interests you.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../docs/RECOVERY.md).

In [ ]:
import sys
from pathlib import Path

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'lab_support/runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from lab_support import checks as check
from lab_support.arrival_files import publish_arrival
from lab_support.checks import todo
from lab_support.runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path
from lab_support.workspace import Workspace

workspace = Workspace(solutions=False)
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="exercise-2"></a>
## Your task

**How can differently written product keys match reliably?**

Core budget: about 5 minutes.

Make a reusable `product_key(column)` function: trim spaces and uppercase the key. Then create `clean_products(raw)` so it returns `product_id` and `category`, with trimmed lowercase categories. Use `product_key` inside it.

Success examples: `" b1 "` → `B1`, `"g1"` → `G1`, `" Books "` → `books`. Keep the original inputs available.

`F` is the alias from `from pyspark.sql import functions as F`. These functions build Spark expressions. `F.col(...)` selects a column; `F.lit(...)` describes a literal value.

The string functions you need for this exercise are listed in the [API reference](../docs/API-REFERENCE.md) under **String operations**.

### Your code — the key expression

In [ ]:
def product_key(column: Column) -> Column:
    """Build the normalised product-key expression."""
    return todo("2: trim the Column, then uppercase it")

### Supplied — apply your helper to the lookup

This wrapper is supplied. It uses your `product_key` function.

In [ ]:
def clean_products(raw: DataFrame) -> DataFrame:
    """Normalise product keys and category names while retaining the lookup row grain."""
    return raw.select(
        product_key(F.col("product_id")).alias("product_id"),
        F.lower(F.trim("category")).alias("category"),
    )

In [ ]:
products = clean_products(raw_products)
products.orderBy("product_id").show()
raw.select("product_id", product_key(F.col("product_id")).alias("clean_key")).show()

### Check

The helper verifies your expression and the lookup values. The raw data must remain unchanged.

In [ ]:
check.keys(raw, products, product_key)

Slide reminder: [Clean the keys](https://dannyscodecorner.github.io/mastering-pyspark/#pyspark-cleaning).

<details>
<summary>Need a nudge? Hint 1</summary>

Use the built-in string functions for trimming and changing case.

</details>

<details>
<summary>A little more help: Hint 2</summary>

Compose the operations from the inside out. Use `.alias(...)` when a derived expression needs a predictable output column name.

</details>

If you need to catch up during class, use the explicit [recovery step](../docs/RECOVERY.md#exercise-2). [Worked solution](../solutions/02-clean-keys.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 5 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### A Python helper is not necessarily a Python UDF

Calling `product_key` constructs a `Column` expression in Python. Spark evaluates its built-in operations on the data. There is no custom Python row function or UDF here.

`select` creates a new DataFrame description. It does not alter `raw`. The next optional task lets you compare their schemas and try renaming/dropping columns.

---
### Expressions and immutability

From `raw`, make an `exploration` DataFrame that adds a normalised `clean_key`, renames `amount_raw` to `source_amount`, and drops `sold_at_raw`. Inspect both schemas. Keep `raw` unchanged.

In [ ]:
exploration = todo("Add a derived column, rename one, and drop another in a separate DataFrame")

In [ ]:
check.projection(raw, exploration)

<details><summary>Hint</summary>

Try `withColumn`, `withColumnRenamed` and `drop`. Keep their returned DataFrame under a new name.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
workspace.save(product_key, clean_products)
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Next: [Exercise 3 — Validate the sales](03-validate.ipynb).

Want more on this topic? You can open these now, using the same saved work: [Reshaping product tags](deeper/product-tags.ipynb).

After your attempt, compare the separate [worked solution](../solutions/02-clean-keys.ipynb).